# Segmentación Avanzada y Customer LTV

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Calcular Customer Lifetime Value** con modelos SQL avanzados
2. **Segmentar clientes** por valor y comportamiento
3. **Proyectar valor futuro** con `SNOWFLAKE.ML.FORECAST`
4. **Aplicar segmentación RFM** (Recencia, Frecuencia, Valor Monetario)
5. **Generar perfiles de segmento** con `CORTEX.COMPLETE()`

---

## Lo Que Construirás

Un **sistema de segmentación y LTV** bancario:
- Cálculo de LTV histórico y proyectado
- Segmentación RFM con NTILE
- Perfiles de segmento generados por IA
- Dashboard de valor de cartera

**Valor de Negocio**: Asignar recursos comerciales según el valor de cada cliente.

---

## Funcionalidades Clave

- `SNOWFLAKE.ML.FORECAST` — Proyección de ingresos por cliente
- `CORTEX.COMPLETE()` — Perfiles de segmento narrativos
- `NTILE()` — Segmentación por cuartiles/quintiles
- Window functions — Cálculos RFM

¡Comencemos!

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS SEGMENTACION_LTV_DB;
CREATE SCHEMA IF NOT EXISTS SEGMENTACION_LTV_DB.PUBLIC;
USE SCHEMA SEGMENTACION_LTV_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS db;

---

## Paso 2: Generar Datos de Clientes

### Qué Vamos a Crear

- **4.000 clientes** con historial de 24 meses
- **Ingresos mensuales por comisiones, intereses y productos** por cliente
- **Datos transaccionales** para cálculo RFM

In [ ]:
-- Clientes con perfil
CREATE OR REPLACE TABLE CLIENTES_LTV (
    cliente_id VARCHAR(10) PRIMARY KEY,
    fecha_alta DATE,
    segmento VARCHAR(20),
    productos_contratados INTEGER,
    saldo_medio DECIMAL(12,2)
);

INSERT INTO CLIENTES_LTV
SELECT
    'CLI' || LPAD(SEQ4()::STRING, 5, '0'),
    DATEADD('day', -UNIFORM(365, 3650, RANDOM()), CURRENT_DATE()),
    ARRAY_CONSTRUCT('Premium','Particular','Joven','Empresa')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    UNIFORM(1, 8, RANDOM()),
    ROUND(UNIFORM(500, 500000, RANDOM()), 2)
FROM TABLE(GENERATOR(ROWCOUNT => 4000));

-- Ingresos mensuales por cliente (24 meses)
CREATE OR REPLACE TABLE INGRESOS_MENSUALES (
    cliente_id VARCHAR(10),
    mes DATE,
    comisiones DECIMAL(10,2),
    intereses DECIMAL(10,2),
    seguros DECIMAL(10,2),
    otros_ingresos DECIMAL(10,2),
    PRIMARY KEY (cliente_id, mes)
);

INSERT INTO INGRESOS_MENSUALES
SELECT
    c.cliente_id,
    DATEADD('month', -m.n, DATE_TRUNC('month', CURRENT_DATE())),
    ROUND(UNIFORM(5, 100, RANDOM()) * c.productos_contratados / 3.0, 2),
    ROUND(c.saldo_medio * UNIFORM(1, 5, RANDOM()) / 1200.0, 2),
    ROUND(UNIFORM(0, 50, RANDOM()), 2),
    ROUND(UNIFORM(0, 20, RANDOM()), 2)
FROM CLIENTES_LTV c
CROSS JOIN (SELECT SEQ4() AS n FROM TABLE(GENERATOR(ROWCOUNT => 24))) m;

SELECT COUNT(DISTINCT cliente_id) AS clientes, COUNT(*) AS registros FROM INGRESOS_MENSUALES;

---

## Paso 3: Calcular LTV y Segmentación RFM

### Customer Lifetime Value

LTV = Ingresos totales generados por el cliente durante su relación.

### Segmentación RFM

- **Recencia (R)**: Días desde la última actividad generadora de ingreso
- **Frecuencia (F)**: Nº de meses con actividad
- **Monetario (M)**: Ingreso total acumulado

Cada dimensión se divide en quintiles (1-5) usando `NTILE(5)`.

In [ ]:
-- Calcular LTV y RFM
CREATE OR REPLACE TABLE SEGMENTACION_LTV AS
WITH metricas AS (
    SELECT
        i.cliente_id,
        SUM(i.comisiones + i.intereses + i.seguros + i.otros_ingresos) AS ltv_historico,
        AVG(i.comisiones + i.intereses + i.seguros + i.otros_ingresos) AS ingreso_medio_mensual,
        DATEDIFF('day', MAX(i.mes), CURRENT_DATE()) AS dias_desde_ultimo_ingreso,
        COUNT(CASE WHEN (i.comisiones + i.intereses) > 10 THEN 1 END) AS meses_activos,
        MAX(i.comisiones + i.intereses + i.seguros + i.otros_ingresos) AS ingreso_max_mensual
    FROM INGRESOS_MENSUALES i GROUP BY 1
)
SELECT
    m.cliente_id,
    c.segmento,
    c.productos_contratados,
    c.saldo_medio,
    DATEDIFF('month', c.fecha_alta, CURRENT_DATE()) AS meses_como_cliente,
    m.ltv_historico,
    m.ingreso_medio_mensual,
    m.dias_desde_ultimo_ingreso,
    m.meses_activos,
    NTILE(5) OVER (ORDER BY m.dias_desde_ultimo_ingreso ASC) AS score_recencia,
    NTILE(5) OVER (ORDER BY m.meses_activos DESC) AS score_frecuencia,
    NTILE(5) OVER (ORDER BY m.ltv_historico DESC) AS score_monetario,
    score_recencia + score_frecuencia + score_monetario AS rfm_total,
    CASE
        WHEN score_recencia + score_frecuencia + score_monetario >= 13 THEN 'VIP'
        WHEN score_recencia + score_frecuencia + score_monetario >= 10 THEN 'Alto Valor'
        WHEN score_recencia + score_frecuencia + score_monetario >= 7 THEN 'Medio'
        ELSE 'Bajo Valor'
    END AS segmento_valor
FROM metricas m
JOIN CLIENTES_LTV c ON m.cliente_id = c.cliente_id;

SELECT segmento_valor, COUNT(*) AS n, 
    ROUND(AVG(ltv_historico),0) AS ltv_medio,
    ROUND(AVG(ingreso_medio_mensual),2) AS ing_mensual_medio
FROM SEGMENTACION_LTV GROUP BY 1 ORDER BY ltv_medio DESC;

---

## Paso 4: Proyectar LTV con Forecast

### ML.FORECAST para Ingresos

Proyectamos el ingreso futuro de cada segmento a 12 meses.

In [ ]:
-- Preparar datos para forecast por segmento
CREATE OR REPLACE TABLE INGRESOS_POR_SEGMENTO AS
SELECT
    i.mes AS fecha,
    s.segmento_valor AS segmento,
    SUM(i.comisiones + i.intereses + i.seguros + i.otros_ingresos) AS ingreso_total
FROM INGRESOS_MENSUALES i
JOIN SEGMENTACION_LTV s ON i.cliente_id = s.cliente_id
GROUP BY 1, 2;

CREATE OR REPLACE SNOWFLAKE.ML.FORECAST forecast_ltv(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'INGRESOS_POR_SEGMENTO'),
    TIMESTAMP_COLNAME => 'FECHA',
    TARGET_COLNAME => 'INGRESO_TOTAL',
    SERIES_COLNAME => 'SEGMENTO'
);

CALL forecast_ltv!FORECAST(FORECASTING_PERIODS => 12);

---

## Paso 5: Perfiles de Segmento con IA

Generamos descripciones narrativas de cada segmento.

In [ ]:
-- Perfiles narrativos por segmento
CREATE OR REPLACE TABLE PERFILES_SEGMENTO AS
WITH stats AS (
    SELECT segmento_valor, COUNT(*) AS n, ROUND(AVG(ltv_historico),0) AS ltv_medio,
        ROUND(AVG(saldo_medio),0) AS saldo_medio, ROUND(AVG(productos_contratados),1) AS prod_medio
    FROM SEGMENTACION_LTV GROUP BY 1
)
SELECT
    segmento_valor, n, ltv_medio, saldo_medio, prod_medio,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        'Describe este segmento bancario en 3 líneas y sugiere 2 acciones comerciales:\n' ||
        'Segmento: ' || segmento_valor || '\n' ||
        'Clientes: ' || n || '\n' ||
        'LTV medio: ' || ltv_medio || '€\n' ||
        'Saldo medio: ' || saldo_medio || '€\n' ||
        'Productos medio: ' || prod_medio || '\n' ||
        'Responde en español.'
    ) AS perfil_narrativo
FROM stats;

SELECT * FROM PERFILES_SEGMENTO;

---

## Paso 6: Dashboard de Valor de Cartera

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Segmentación y Customer Lifetime Value')
st.markdown('### Análisis de Valor de Cartera')

df = session.sql('SELECT * FROM SEGMENTACION_LTV').to_pandas()

total_ltv = df['LTV_HISTORICO'].sum()
vip = len(df[df['SEGMENTO_VALOR'] == 'VIP'])

c1, c2, c3 = st.columns(3)
c1.metric('Total Clientes', f'{len(df):,}')
c2.metric('LTV Total', f'{total_ltv:,.0f}€')
c3.metric('Clientes VIP', f'{vip:,}')

st.markdown('---')
st.subheader('Distribución por Segmento de Valor')
df_seg = df['SEGMENTO_VALOR'].value_counts()
st.bar_chart(df_seg)

st.markdown('---')
st.subheader('Perfiles de Segmento')
df_perf = session.sql('SELECT * FROM PERFILES_SEGMENTO').to_pandas()
for _, row in df_perf.iterrows():
    st.markdown(f"**{row['SEGMENTO_VALOR']}** ({row['N']:,} clientes, LTV: {row['LTV_MEDIO']:,.0f}€)")
    st.write(row['PERFIL_NARRATIVO'])
    st.markdown('---')

---

## Paso 7: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS SEGMENTACION_LTV_DB;